In [10]:
"""
Day 1 - API Data Analysis Using Python Structures
Pesa Pulse - Nairobi Fintech Start-up Analysis

"""
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

API_URL = "https://jsonplaceholder.typicode.com/users"


def get_customers(url: str = API_URL) -> list[dict]:
    """Fetch the customer list with error handling and timeout.
    
    Reports network failures cleanly without raising a traceback.
    """
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"[Network Error] Could not fetch data from {url}: {e}")
        return []

In [11]:
def explore(record: dict, indent: int = 0) -> None:
    """Print every key in a (possibly nested) record with its type, indented by depth."""
    for key, value in record.items():
        prefix = "  " * indent
        print(f"{prefix}- {key}: {type(value).__name__}")
        if isinstance(value, dict):
            explore(value, indent + 1)
        elif isinstance(value, list) and len(value) > 0 and isinstance(value[0], dict):
            for item in value:
                if isinstance(item, dict):
                    explore(item, indent + 1)

In [12]:
def extract_attributes(customers: list[dict]) -> list[dict]:
    """Return one flat dict per customer with required attributes.
    
    Extracts id, name, username, email, email_domain, phone, city, zipcode,
    latitude, longitude, company, and website. Lat/lng are converted to floats.
    """
    flat_rows = []
    for customer in customers:
        email = customer.get("email", "")
        email_domain = email.split(".")[-1] if "." in email else ""
        
        address = customer.get("address", {})
        geo = address.get("geo", {})
        company_obj = customer.get("company", {})
        
        # Safely parse latitude and longitude as floats
        try:
            lat = float(geo.get("lat", 0.0))
        except (ValueError, TypeError):
            lat = 0.0
            
        try:
            lng = float(geo.get("lng", 0.0))
        except (ValueError, TypeError):
            lng = 0.0
            
        row = {
            "customer_id": customer.get("id"),
            "name": customer.get("name"),
            "username": customer.get("username"),
            "email": email,
            "email_domain": email_domain,
            "phone": customer.get("phone"),
            "city": address.get("city"),
            "zipcode": address.get("zipcode"),
            "latitude": lat,
            "longitude": lng,
            "company": company_obj.get("name"),
            "website": customer.get("website"),
        }
        flat_rows.append(row)
    return flat_rows

In [13]:
def unique_cities(customers: list[dict]) -> set[str]:
    """Return the set of distinct cities from customer addresses."""
    return {
        customer["address"]["city"]
        for customer in customers
        if "address" in customer and "city" in customer["address"]
    }




In [14]:
def city_counts(customers: list[dict]) -> dict[str, int]:
    """Count customers per city using a loop and dict, sorted by count descending."""
    cities = [
        customer["address"]["city"]
        for customer in customers
        if "address" in customer and "city" in customer["address"]
    ]
    counts = {}
    for city in cities:
        counts[city] = counts.get(city, 0) + 1
        
    return dict(sorted(counts.items(), key=lambda item: (-item[1], item[0])))




In [15]:
def email_domain_counts(customers: list[dict]) -> dict[str, int]:
    """Count customers per email top-level domain using a dict comprehension."""
    domains = [
        customer["email"].split(".")[-1]
        for customer in customers
        if "email" in customer and "." in customer["email"]
    ]
    raw_counts = {dom: domains.count(dom) for dom in set(domains)}
    return dict(sorted(raw_counts.items(), key=lambda item: (-item[1], item[0])))




In [16]:
def to_dataframe(rows: list[dict]) -> pd.DataFrame:
    """Convert flat records into a pandas DataFrame and print summary info."""
    df = pd.DataFrame(rows)
    print("\n--- DataFrame Inspection ---")
    print(f"Shape : {df.shape}")
    print("\nData Types:")
    print(df.dtypes)
    print("\nPandas City Value Counts (Validation):")
    print(df["city"].value_counts())
    return df




In [18]:
def plot_distribution(counts: dict[str, int], path: Path) -> None:
    """Save a labelled bar chart of the distribution to path."""
    plt.figure(figsize=(10, 6))
    categories = list(counts.keys())
    values = list(counts.values())
    
    bars = plt.bar(categories, values, color="#10b981", width=0.45, edgecolor="none")
    plt.xlabel("Email Top-Level Domain (TLD)", fontsize=11, fontweight="bold", labelpad=10)
    plt.ylabel("Customer Count", fontsize=11, fontweight="bold", labelpad=10)
    plt.title("Pesa Pulse - Customer Distribution by Email Domain", fontsize=13, fontweight="bold", pad=15)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    
    for bar in bars:
        height = bar.get_height()
        plt.annotate(
            f"{height}",
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold"
        )
        
    plt.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"Chart saved to      : {path}")




In [19]:
if __name__ == "__main__":
    print("=== Pesa Pulse Customer Data Pipeline ===")
    
    # 1. Retrieve data
    customers = get_customers(API_URL)
    if not customers:
        print("Pipeline aborted due to API retrieval failure.")
        exit(1)
        
    # 2. Explore nested structure
    print("\n--- Nested Structure Exploration (First Record) ---")
    explore(customers[0])
    
    # 3. Extract attributes
    rows = extract_attributes(customers)
    
    # 4. Unique cities analysis
    cities = unique_cities(customers)
    num_customers = len(customers)
    num_cities = len(cities)
    
    # 5. Hand-written aggregations
    c_counts = city_counts(customers)
    d_counts = email_domain_counts(customers)
    
    most_common_city, most_common_count = list(c_counts.items())[0]
    
    # 6. DataFrame conversion
    df = to_dataframe(rows)
    
    # 7. Export CSVs
    output_dir = Path("output")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    customers_csv_path = output_dir / "customers.csv"
    city_counts_csv_path = output_dir / "city_counts.csv"
    
    df.to_csv(customers_csv_path, index=False)
    
    city_df = pd.DataFrame(list(c_counts.items()), columns=["city", "customer_count"])
    city_df.to_csv(city_counts_csv_path, index=False)
    
    # 8. Visualization
    chart_path = output_dir / "customer_distribution.png"
    plot_distribution(d_counts, chart_path)
    
    # Console Findings Summary
    print("\n--- Console Findings ---")
    print(f"Customers retrieved : {num_customers}")
    print(f"Unique cities       : {num_cities}")
    print(f"Most common city    : {most_common_city} ({most_common_count})")
    print(f"Email domains       : {d_counts}")
    print(f"Chart saved to      : {chart_path}")

=== Pesa Pulse Customer Data Pipeline ===

--- Nested Structure Exploration (First Record) ---
- id: int
- name: str
- username: str
- email: str
- address: dict
  - street: str
  - suite: str
  - city: str
  - zipcode: str
  - geo: dict
    - lat: str
    - lng: str
- phone: str
- website: str
- company: dict
  - name: str
  - catchPhrase: str
  - bs: str

--- DataFrame Inspection ---
Shape : (10, 12)

Data Types:
customer_id       int64
name                str
username            str
email               str
email_domain        str
phone               str
city                str
zipcode             str
latitude        float64
longitude       float64
company             str
website             str
dtype: object

Pandas City Value Counts (Validation):
city
Gwenborough       1
Wisokyburgh       1
McKenziehaven     1
South Elvis       1
Roscoeview        1
South Christy     1
Howemouth         1
Aliyaview         1
Bartholomebury    1
Lebsackbury       1
Name: count, dtype: int64
Chart sa